# Final model selection and test evaluation

This notebook performs the **final** model workflow.

- With **one input model**, it loads or computes test probabilities and evaluates the frozen model on the test set.
- With **two input models**, it assumes one metadata model and one image model, selects the late-fusion rule on validation using patient-grouped cross-validation, freezes the selected fusion and clinical threshold, creates a new fusion-model directory, and only then evaluates it on test.
- Base models are **never retrained**.
- Cached `validation_predictions.parquet` / `test_predictions.parquet` files are reused whenever available.

## Methodological rules

Fusion candidates:

1. simple average;
2. weighted average;
3. logistic stacking.

Fusion selection uses **5-fold `StratifiedGroupKFold` grouped by `patient_id`**. The comparison hierarchy is:

1. PR-AUC — higher is better;
2. ROC-AUC — higher is better;
3. Brier score — lower is better.

After selecting the fusion method from out-of-fold validation predictions, that method is refitted on the complete validation set. The clinical threshold is then selected on validation with the same project rule: maximum specificity while maintaining the requested minimum sensitivity.

The test set is not used for any model, fusion or threshold selection.

In [6]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from skin_lesion_ai.inference.final_models import run_final_model_pipeline

## Configuration

In [ ]:
# ---------------------------------------------------------------------
# INPUT MODELS
# ---------------------------------------------------------------------
# Provide exactly:
# - one model directory, or
# - two model directories: one metadata model + one image model.
#
# Paths may be absolute or relative to the repository root.

MODEL_DIRECTORIES = [
    "data/models/xgboost_metadata_h1",
    "data/models/efficientnet_b0_image_h1_full_train",
]

# ---------------------------------------------------------------------
# FINAL-EVALUATION SETTINGS
# ---------------------------------------------------------------------

TARGET_SENSITIVITY = 0.95

N_SPLITS = 5
RANDOM_STATE = 42

# Weighted-average candidates: 0.00, 0.01, ..., 1.00.
WEIGHT_GRID_STEP = 0.01

# "auto" selects CUDA -> MPS -> CPU for direct PyTorch image inference.
DEVICE = "auto"

# Normally keep False. When True, cached validation/test probabilities
# are ignored and inference is rerun. Base models are still never retrained.
FORCE_RECOMPUTE_PREDICTIONS = False

# Optional fallback for legacy models whose input/preprocessing cannot be
# inferred safely from the saved model and metadata.
#
# Keys can be the model directory name or its path.
# Leave empty unless the pipeline explicitly asks for missing information.
#
# Example:
# MODEL_OVERRIDES = {
#     "old_metadata_model_h1": {
#         "input_kind": "metadata",
#         "feature_columns": ["age_approx", "sex", "..."],
#     },
#     "old_image_model_h1": {
#         "input_kind": "direct_image",
#         "architecture": "efficientnet_b0",
#         "image_size": 136,
#         "dropout": 0.2,
#         "normalization": "imagenet",
#         "output_type": "logit",
#     },
# }

MODEL_OVERRIDES = {}

MIN_PR_AUC_GAIN = 0.01

## Run final pipeline

In [ ]:
if len(MODEL_DIRECTORIES) not in {1, 2}:
    raise ValueError(
        "Set MODEL_DIRECTORIES to exactly one model, or to two models "
        "(one metadata model + one image model)."
    )

results = run_final_model_pipeline(
    model_directories=MODEL_DIRECTORIES,
    target_sensitivity=TARGET_SENSITIVITY,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    weight_grid_step=WEIGHT_GRID_STEP,
    min_pr_auc_gain=MIN_PR_AUC_GAIN,
    model_overrides=MODEL_OVERRIDES,
    force_recompute_predictions=FORCE_RECOMPUTE_PREDICTIONS,
    device=DEVICE,
)

print(f"Mode: {results['mode']}")
print(f"Hypothesis: H{results['hypothesis']}")
print(f"Final model directory: {results['final_model_directory']}")
print(f"Frozen threshold: {results['selected_threshold']:.8f}")

if results["mode"] == "fusion":
    print(f"Selected fusion: {results['selected_method']}")
    display(results["fusion_cv_results"])

Using cached validation predictions: /Users/carlesraichbros/my-image-classifier/data/models/xgboost_metadata_h1/validation_predictions.parquet
Using cached validation predictions: /Users/carlesraichbros/my-image-classifier/data/models/efficientnet_b0_image_h1_full_train/validation_predictions.parquet


/Users/carlesraichbros/my-image-classifier/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/carlesraichbros/my-image-classifier/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warni


Validation candidate comparison:
              method candidate_type   pr_auc  roc_auc  brier_score  best_fusion  selected_overall
       metadata_only     base_model 0.053664 0.798312     0.002619        False              True
    weighted_average         fusion 0.029666 0.796388     0.002656         True             False
      simple_average         fusion 0.026164 0.835592     0.034951        False             False
          image_only     base_model 0.025260 0.831656     0.133183        False             False
interaction_stacking         fusion 0.024484 0.825713     0.002611        False             False
   logistic_stacking         fusion 0.024377 0.826388     0.002612        False             False
Using existing test evaluation: /Users/carlesraichbros/my-image-classifier/data/models/xgboost_metadata_h1/test
Using cached test predictions: /Users/carlesraichbros/my-image-classifier/data/models/efficientnet_b0_image_h1_full_train/test_predictions.parquet

Validation vs test c

## Final test metrics

In [9]:
display(results["test_results"]["summary"].T)

print(
    "Final test JSON:",
    results["test_results"]["metrics_json_path"],
)
print(
    "Test output directory:",
    results["test_results"]["output_directory"],
)

,0
total_lesions,38125.000000
total_patients,99.000000
positive_lesions,100.000000
negative_lesions,38025.000000
patients_with_positive_lesion,64.000000
lesion_prevalence,0.002623
patient_prevalence,0.646465
pr_auc,0.085158
roc_auc,0.839040
brier_score,0.002589


Final test JSON: /Users/carlesraichbros/my-image-classifier/data/models/xgboost_metadata_h1/test/metrica_test_final.json
Test output directory: /Users/carlesraichbros/my-image-classifier/data/models/xgboost_metadata_h1/test


## Expected outputs

For a single model:

```text
<model_directory>/
├── validation_predictions.parquet      # reused/created only if needed
├── test_predictions.parquet            # inference cache
└── test/
    ├── metrica_test_final.json
    ├── test_metrics.csv
    ├── test_predictions.parquet
    ├── precision_recall_curve.jpg
    ├── roc_curve.jpg
    ├── pr_roc_curves.jpg
    └── clinical_summary.jpg
```

For a two-model fusion:

```text
data/models/fusion_<metadata>__<image>/
├── fusion_model.joblib
├── model_metadata.json
├── fusion_metadata.json
├── fusion_cv_results.csv
├── validation_predictions.parquet
├── test_predictions.parquet
└── test/
    ├── metrica_test_final.json
    ├── test_metrics.csv
    ├── test_predictions.parquet
    └── ...
```

`fusion_cv_results.csv` keeps only the compact OOF comparison required to document why the winning fusion method was selected; full evaluation plots are generated only for the final frozen model.

In [10]:
from skin_lesion_ai.utils.data_utils import get_project_root
from skin_lesion_ai.inference.evaluation import compute_threshold_free_metrics
from skin_lesion_ai.inference.final_models import _load_split

root = get_project_root()

metadata_model_dir = root / "data/models/xgboost_metadata_h1"
image_model_dir = root / "data/models/efficientnet_b0_image_h1_full_train"

validation_split = _load_split(
    hypothesis=1,
    split="validation",
)

metadata_validation_predictions = pd.read_parquet(
    metadata_model_dir / "validation_predictions.parquet"
)

image_validation_predictions = pd.read_parquet(
    image_model_dir / "validation_predictions.parquet"
)

df_check = (
    validation_split[["isic_id", "target_biopsy"]]
    .merge(
        metadata_validation_predictions.rename(columns={"probability": "p_meta"}),
        on="isic_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        image_validation_predictions.rename(columns={"probability": "p_image"}),
        on="isic_id",
        how="inner",
        validate="one_to_one",
    )
)

y = df_check["target_biopsy"].to_numpy()

print(
    "Metadata:",
    compute_threshold_free_metrics(
        y_true=y,
        y_prob=df_check["p_meta"].to_numpy(),
    ),
)

print(
    "EfficientNet:",
    compute_threshold_free_metrics(
        y_true=y,
        y_prob=df_check["p_image"].to_numpy(),
    ),
)

print("Rows:", len(df_check))

Metadata: {'pr_auc': 0.053663978615571, 'roc_auc': 0.7983115206077536, 'brier_score': 0.0026192367493966828}
EfficientNet: {'pr_auc': 0.025260192461083658, 'roc_auc': 0.8316558318992563, 'brier_score': 0.1331828705434983}
Rows: 38131


In [11]:
import pandas as pd

from skin_lesion_ai.inference.evaluation import (
    compute_threshold_free_metrics,
)
from skin_lesion_ai.inference.final_models import _load_split


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

metadata_model_dir = Path("data/models/xgboost_metadata_h1")

image_model_dir = Path("data/models/efficientnet_b0_image_h1_full_train")


# ------------------------------------------------------------
# Load validation split and cached predictions
# ------------------------------------------------------------

validation_split = _load_split(
    hypothesis=1,
    split="validation",
)

metadata_validation_predictions = pd.read_parquet(
    metadata_model_dir / "validation_predictions.parquet"
)

image_validation_predictions = pd.read_parquet(
    image_model_dir / "validation_predictions.parquet"
)


# ------------------------------------------------------------
# Align by isic_id
# ------------------------------------------------------------

df_check = (
    validation_split[["isic_id", "target_biopsy"]]
    .merge(
        metadata_validation_predictions.rename(columns={"probability": "p_meta"}),
        on="isic_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        image_validation_predictions.rename(columns={"probability": "p_image"}),
        on="isic_id",
        how="inner",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

y = df_check["target_biopsy"].to_numpy()

meta_metrics = compute_threshold_free_metrics(
    y_true=y,
    y_prob=df_check["p_meta"].to_numpy(),
)

image_metrics = compute_threshold_free_metrics(
    y_true=y,
    y_prob=df_check["p_image"].to_numpy(),
)

print("Metadata:")
print(meta_metrics)

print("\nEfficientNet:")
print(image_metrics)

print("\nRows:", len(df_check))

FileNotFoundError: [Errno 2] No such file or directory: 'data/models/xgboost_metadata_h1/validation_predictions.parquet'